# CS690A Project 3 - Phase 2 prompt runner for Method 3 (m3_identity_program)
Reads the phase-1 folder, anonymises niches, builds T1-T4, calls Gemini, writes `results_m3_identity_program.csv`.

In [1]:
IN_DIR = "/content/outputs/m3_identity_program"  # adjustable
ZIP_PATH = "/content/m3_identity_program.zip"  # adjustable
RUN_DIR = "/content/outputs/m3_prompt_runner"  # adjustable
METHOD_ID = "m3_identity_program"
DRY_RUN = False  # adjustable

MODEL = "models/gemini-3.5-flash"  # adjustable
API_VERSION = "v1"
TEMPERATURE = 1.0  # adjustable
MAX_OUTPUT_TOKENS = 8192
N_RUNS = 5  # adjustable
TEMPLATES = ["T1", "T2", "T3", "T4"]

TOP_N = 10  # adjustable
LR_TOP = 5  # adjustable
SHUFFLE_SEED = 7
TISSUE_CONTEXT = "human tumour tissue profiled with NanoString CosMx SMI, 960-gene panel"  # adjustable

SLEEP_RANGE = (1, 5)  # adjustable
MAX_RETRIES = 6  # adjustable
BACKOFF_BASE_S = 2  # adjustable
REQUEST_TIMEOUT_S = 300  # adjustable

EXPERT_REGION_NAMES = ["myeloid-enriched stroma", "stroma", "tumor interior", "lymphoid structure",
    "tumor-stroma boundary", "neutrophils", "immune", "macrophages"]
CONFIG_KEYS = [k for k in list(globals()) if k.isupper() and not k.startswith("_")]

In [2]:
import os, re, json, time, random, string, shutil, zipfile, datetime
import numpy as np, pandas as pd, requests

def sorted_niches(vals):
    u = [str(x) for x in pd.unique(pd.Series(vals).astype(str))]
    return sorted(u, key=lambda x: (not x.isdigit(), int(x) if x.isdigit() else 0, x))
def save_json(path, obj):
    with open(path, "w") as f: json.dump(obj, f, indent=2, default=str)

NEEDED = ["identity_markers.csv", "program_markers.csv", "niche_LR.csv", "niche_summary.csv"]
if not all(os.path.exists(os.path.join(IN_DIR, f)) for f in NEEDED):
    if not os.path.exists(ZIP_PATH):
        from google.colab import files
        up = files.upload(); ZIP_PATH = "/content/" + next(k for k in up if k.endswith(".zip"))
    os.makedirs(IN_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(IN_DIR)
    for f in NEEDED:
        if not os.path.exists(os.path.join(IN_DIR, f)):
            hit = [os.path.join(r, f) for r, _, fs in os.walk(IN_DIR) if f in fs]
            if hit: shutil.copy(hit[0], os.path.join(IN_DIR, f))
missing = [f for f in NEEDED if not os.path.exists(os.path.join(IN_DIR, f))]
assert not missing, f"missing in {IN_DIR}: {missing}"

idm = pd.read_csv(os.path.join(IN_DIR, "identity_markers.csv"), dtype={"niche": str})
prg = pd.read_csv(os.path.join(IN_DIR, "program_markers.csv"), dtype={"niche": str})
lrp = pd.read_csv(os.path.join(IN_DIR, "niche_LR.csv"), dtype={"niche": str, "source_niche": str})
summ = pd.read_csv(os.path.join(IN_DIR, "niche_summary.csv"), dtype={"niche": str})
for df, cols in [(idm, ["niche", "rank", "gene", "log2FC"]), (prg, ["niche", "rank", "gene", "log2FC"]),
                 (lrp, ["niche", "ligand", "receptor", "source_niche", "mean_score"]), (summ, ["niche", "n_cells"])]:
    assert set(cols) <= set(df.columns), f"unexpected columns {list(df.columns)}"

niches = sorted_niches(summ["niche"])
assert len(niches) <= 26
shuf = list(niches); random.Random(SHUFFLE_SEED).shuffle(shuf)
letters = list(string.ascii_uppercase[:len(niches)])
LAST = letters[-1]
letter_of = dict(zip(shuf, letters)); niche_of = dict(zip(letters, shuf))
n_cells_of = dict(zip(summ["niche"], summ["n_cells"]))
os.makedirs(os.path.join(RUN_DIR, "raw"), exist_ok=True); os.makedirs(os.path.join(RUN_DIR, "prompts"), exist_ok=True)
pd.DataFrame({"niche_letter": letters, "niche_id": [niche_of[l] for l in letters]}).to_csv(os.path.join(RUN_DIR, "niche_letter_key.csv"), index=False)

forbidden = set(EXPERT_REGION_NAMES)
lab_path = os.path.join(IN_DIR, "labels.csv")
if os.path.exists(lab_path): forbidden |= set(pd.read_csv(lab_path, usecols=["Region"])["Region"].astype(str).unique())
print(f"{len(niches)} niches -> letters A-{LAST}; {len(forbidden)} forbidden region names")

7 niches -> letters A-G; 8 forbidden region names


In [3]:
def gene_line(df, niche, with_lfc):
    d = df[df["niche"] == niche].sort_values("rank").head(TOP_N)
    if not len(d): return "none"
    return ", ".join(f"{g} ({l:+.2f})" if with_lfc else str(g) for g, l in zip(d["gene"], d["log2FC"]))

def lr_line(niche):
    d = lrp[lrp["niche"] == niche].sort_values("mean_score", ascending=False).head(LR_TOP)
    d = d[d["source_niche"].isin(letter_of)]
    if not len(d): return "none"
    return "; ".join(f"{a} -> {b} (sender {letter_of[s]}, score {m:.2f})" for a, b, s, m in zip(d["ligand"], d["receptor"], d["source_niche"], d["mean_score"]))

def block(title, fn):
    return f"### {title}:\n" + "\n".join(f"{l}: {fn(niche_of[l])}" for l in letters) + "\n"

JSON_FMT = ("### Response format:\nEnd your answer with a final JSON array in a fenced ```json block, one object per region A-" + LAST + ":\n"
    "```json\n[{\"region\": \"A\", \"label\": \"...\", \"confidence\": \"high|medium|low\", \"justification\": \"...\"}]\n```\n")
COT = ("### Reasoning:\nBefore the JSON, reason step by step: (1) from each region's identity genes, infer the likely cell types; "
    "(2) from its program genes and ligand-receptor pairs, infer its functional state; (3) combine these into one niche label.\n\n")

def build_prompt(t):
    p = ("You are an expert computational biologist and spatial transcriptomics specialist.\n\n### Task:\n"
         "Identify and annotate the biological cellular niche (tissue microenvironment identity) for each region below.\n\n### Metadata:\n"
         "- Modality/Platform: NanoString CosMx SMI (960-gene panel)\n" f"- Number of regions: {len(letters)} (A-{LAST})\n- Target Species: Human\n")
    if t != "T1": p += f"- Tissue context: {TISSUE_CONTEXT}\n"
    p += "\n"
    if t == "T1":
        p += block("Marker genes per region", lambda n: gene_line(idm, n, False))
    elif t == "T2":
        p += block("Identity marker genes per region (what the cells are)", lambda n: gene_line(idm, n, False)) + "\n"
        p += block("Program marker genes per region (what the niche is doing)", lambda n: gene_line(prg, n, False))
    else:
        p += block("Identity marker genes per region, with log2FC (what the cells are)", lambda n: gene_line(idm, n, True)) + "\n"
        p += block("Program marker genes per region, with log2FC (what the niche is doing)", lambda n: gene_line(prg, n, True)) + "\n"
        p += block("Ligand-receptor pairs per region (the region is the receiver)", lr_line)
    p += "\n" + (COT if t == "T4" else "") + JSON_FMT
    return p

def check_prompt(p):
    assert "Region" not in p, "prompt contains 'Region'"
    assert "banksy" not in p.lower(), "prompt contains 'BANKSY'"
    for name in forbidden:
        assert not re.search(r"\b" + re.escape(name) + r"\b", p, re.I), f"prompt contains region name: {name}"

PROMPTS = {t: build_prompt(t) for t in TEMPLATES}
for t, p in PROMPTS.items():
    check_prompt(p)
    open(os.path.join(RUN_DIR, "prompts", f"{t}.txt"), "w").write(p)
print("prompt checks passed for", list(PROMPTS))
if DRY_RUN:
    for t, p in PROMPTS.items(): print(f"\n{'=' * 20} {t} {'=' * 20}\n{p}")

prompt checks passed for ['T1', 'T2', 'T3', 'T4']


In [4]:
class StopRun(Exception): pass

def call_gemini(prompt, key):
    url = f"https://generativelanguage.googleapis.com/{API_VERSION}/{MODEL}:generateContent"
    body = {"contents": [{"parts": [{"text": prompt}]}], "generationConfig": {"temperature": TEMPERATURE, "maxOutputTokens": MAX_OUTPUT_TOKENS}}
    for k in range(MAX_RETRIES + 1):
        wait = BACKOFF_BASE_S * 2 ** k + random.uniform(0, 1)
        try:
            r = requests.post(url, headers={"x-goog-api-key": key, "Content-Type": "application/json"}, json=body, timeout=REQUEST_TIMEOUT_S)
        except (requests.ConnectionError, requests.Timeout) as e:
            print(f"connection error ({type(e).__name__}), retry in {wait:.0f}s"); time.sleep(wait); continue
        if r.status_code in (500, 503):
            print(f"HTTP {r.status_code}, retry in {wait:.0f}s"); time.sleep(wait); continue
        if r.status_code == 429: raise StopRun("HTTP 429 (quota/rate limit)")
        if r.status_code != 200: raise StopRun(f"HTTP {r.status_code}: {r.text[:400]}")
        return r.json()
    raise StopRun("retries exhausted")

def parse_response(data):
    cand = (data.get("candidates") or [{}])[0]
    parts = (cand.get("content") or {}).get("parts") or []
    text = "".join(p.get("text", "") for p in parts if not p.get("thought"))
    finish = cand.get("finishReason") or (data.get("promptFeedback") or {}).get("blockReason")
    return text, finish, data.get("modelVersion"), data.get("usageMetadata")

def raw_path(t, run): return os.path.join(RUN_DIR, "raw", f"{t}_run{run}.json")
pending = [(t, r) for r in range(1, N_RUNS + 1) for t in TEMPLATES if not os.path.exists(raw_path(t, r))]
print(f"{len(pending)} of {len(TEMPLATES) * N_RUNS} (template, run) calls pending")
stop_reason = None
if DRY_RUN:
    print("DRY_RUN=True: no API calls made")
else:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
    for t, r in list(pending):
        try:
            data = call_gemini(PROMPTS[t], key)
        except StopRun as e:
            stop_reason = str(e); print("STOPPED:", stop_reason); break
        text, finish, ver, usage = parse_response(data)
        save_json(raw_path(t, r), {"template": t, "run": r, "model": MODEL, "model_version": ver, "finish_reason": finish,
            "temperature": TEMPERATURE, "usage": usage, "text": text})
        pending.remove((t, r)); print(f"saved {t} run {r} finish={finish}")
        time.sleep(random.uniform(*SLEEP_RANGE))
    print("remaining:", pending if pending else "none")

20 of 20 (template, run) calls pending
saved T1 run 1 finish=STOP
saved T2 run 1 finish=STOP
saved T3 run 1 finish=STOP
saved T4 run 1 finish=STOP
saved T1 run 2 finish=STOP
saved T2 run 2 finish=STOP
saved T3 run 2 finish=STOP
saved T4 run 2 finish=STOP
saved T1 run 3 finish=STOP
saved T2 run 3 finish=STOP
saved T3 run 3 finish=STOP
saved T4 run 3 finish=STOP
saved T1 run 4 finish=STOP
saved T2 run 4 finish=STOP
saved T3 run 4 finish=STOP
saved T4 run 4 finish=STOP
saved T1 run 5 finish=STOP
saved T2 run 5 finish=STOP
saved T3 run 5 finish=STOP
STOPPED: HTTP 429 (quota/rate limit)
remaining: [('T4', 5)]


In [5]:
def parse_json_block(text):
    blocks = re.findall(r"```(?:json)?\s*(.*?)```", text or "", flags=re.S | re.I)
    if not blocks: return None
    try: arr = json.loads(blocks[-1])
    except Exception: return None
    return arr if isinstance(arr, list) else None

COLS = ["method", "template", "run", "model", "model_version", "niche_letter", "niche_id", "n_cells", "label", "confidence",
    "justification", "parse_ok", "finish_reason", "raw_file"]
rows, versions = [], set()
for t in TEMPLATES:
    for r in range(1, N_RUNS + 1):
        p = raw_path(t, r)
        if not os.path.exists(p): continue
        raw = json.load(open(p)); versions.add(raw.get("model_version"))
        arr = parse_json_block(raw.get("text"))
        by = {}
        for o in arr or []:
            if isinstance(o, dict) and str(o.get("region", "")).strip().upper() in letters: by[str(o["region"]).strip().upper()] = o
        for l in letters:
            o = by.get(l); ok = bool(o) and isinstance(o.get("label"), str) and bool(o.get("label").strip())
            rows.append({"method": METHOD_ID, "template": t, "run": r, "model": raw.get("model"), "model_version": raw.get("model_version"),
                "niche_letter": l, "niche_id": niche_of[l], "n_cells": n_cells_of.get(niche_of[l]),
                "label": o.get("label") if ok else "", "confidence": o.get("confidence", "") if ok else "",
                "justification": o.get("justification", "") if ok else "", "parse_ok": ok,
                "finish_reason": raw.get("finish_reason"), "raw_file": os.path.relpath(p, RUN_DIR)})
res_path = os.path.join(RUN_DIR, f"results_{METHOD_ID}.csv")
if rows:
    res = pd.DataFrame(rows, columns=COLS); res.to_csv(res_path, index=False)
    print(res.groupby("template")["parse_ok"].mean().round(2).to_dict(), "| rows:", len(res))
else:
    print("no raw responses yet; results CSV not written")

save_json(os.path.join(RUN_DIR, "run_config.json"), {"config": {k: globals()[k] for k in CONFIG_KEYS},
    "endpoint": f"https://generativelanguage.googleapis.com/{API_VERSION}/{MODEL}:generateContent",
    "model_versions_observed": sorted(str(v) for v in versions if v), "temperature": TEMPERATURE, "max_output_tokens": MAX_OUTPUT_TOKENS,
    "n_letters": len(letters), "letter_range": f"A-{LAST}", "shuffle_seed": SHUFFLE_SEED, "input_dir": IN_DIR, "stopped_reason": stop_reason,
    "pending": [list(x) for x in pending], "written_at": datetime.datetime.now().isoformat(timespec="seconds")})

{'T1': 1.0, 'T2': 1.0, 'T3': 1.0, 'T4': 1.0} | rows: 133


In [6]:
if os.path.exists(res_path):
    shutil.make_archive("/content/m3_prompt_runner_results", "zip", RUN_DIR)
    try:
        from google.colab import files
        files.download("/content/m3_prompt_runner_results.zip")
    except ImportError:
        print("not in Colab; zip at /content/m3_prompt_runner_results.zip")
else:
    print("nothing to zip yet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>